# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nooragab/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nooragab/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check repo root"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship
Starter data found. You're ready.


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

## My Lane: Refresh / Content Opportunity Scoring

I'm choosing Lane 2 — Refresh / Content Opportunity Scoring. I already ran the starter
pipeline (notebooks 01 and 02) and directly observed this exact problem in action: a
hand-written rule baseline scored Precision@50 = 0.240, while a random forest model reached
Precision@50 = 0.740 — roughly 3x better at identifying which pages actually belong in a
"review first" queue. I'm picking this lane because the label (declining content) is already
observable and well-defined in the starter data, I've seen firsthand that a plain rule leaves
real signal on the table, and the output — a ranked review queue with reason codes — maps
directly to an action a content editor could realistically take.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## The Question

**Decision:** For a content editor with limited review capacity, which pages should be
reviewed first for refresh, expansion, or protection?

**Unit of analysis:** one page (content_id), scored using its trailing 90-day window of
observable signals (impressions, clicks, position, freshness, word count, etc.)

**Output:** a ranked queue of pages, ordered by refresh-opportunity score, each with a
reason code (e.g. stale_visible_page, declining_with_demand) explaining why it was flagged.

**Action:** an editor pulls the top N pages from the queue and reviews/refreshes them first,
instead of guessing or working through pages randomly.

**Cost of a wrong call:** a false positive wastes editor time on a page that didn't need
attention; a false negative lets a genuinely declining page keep losing visibility
un-reviewed. Given limited editor hours, precision at the top of the queue matters more
than catching every possible case.

**Why ML, not just a rule?** The starter pipeline showed a hand-written rule reaches only
~24% precision in its top 50 — 3 out of 4 flagged pages weren't actually declining. A model
can combine many weak, tangled signals in ways a hand-written if/else cannot, and this data
has enough signal for that combination to pay off (random forest reached ~74% precision on
the same data).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

### Numbers that show this lane is worth pursuing

I'll pull three numbers from the starter dataset: how big the "declining" population is, how many
pages are both stale and still visible (a realistic reviewable population), and the precision gap
between a hand-written rule and a learned model.

Note: the combined stale + visible + declining filter is intentionally strict, which is why
only 16 of 30,000 pages qualify. This tells me the review population is small on this starter
slice — a signal that thresholds need tuning, and that the full warehouse (with far more
clients and history) will give a more realistic-sized queue to validate this lane on.

In [ ]:
# This cell is for CODE

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

declining_rate = (df["trend_direction"].str.lower() == "down").mean()
print(f"Share of pages currently declining: {declining_rate:.1%}")

stale_visible_declining = df[
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500) &
    (df["trend_direction"].str.lower() == "down")
]
print(f"Stale + visible + declining pages: {len(stale_visible_declining):,} of {len(df):,} total")

print("Hand-written rule Precision@50: 0.240  (~12 of top 50 correct)")
print("Random forest    Precision@50: 0.740  (~37 of top 50 correct)")

Share of pages currently declining: 54.2%
Stale + visible + declining pages: 16 of 30,000 total
Hand-written rule Precision@50: 0.240  (~12 of top 50 correct)
Random forest    Precision@50: 0.740  (~37 of top 50 correct)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What I can and can't claim

**I can say:** this lane produces an observed, decision-support ranking — a prioritized
list of pages an editor could review first, backed by measurable Precision@K on held-out
clients. I can say a learned model outperformed a hand-written rule on this specific
starter slice, and describe that as directional evidence worth testing at larger scale.

**I cannot say:** that a page flagged as "declining" is guaranteed to recover if refreshed
— that would require a causal experiment (e.g. A/B testing an actual refresh), not just a
ranking model. I cannot claim I've reverse-engineered a Google ranking factor, and I cannot
claim any result generalizes beyond this anonymized 30,000-row starter slice until it's
re-validated on the full warehouse. Any client-identifying detail (names, URLs, raw queries)
will never appear in my write-up — only pseudonymized ids and aggregated metrics.

In [ ]:
sensitive_terms = ["url", "query", "keyword", "domain", "client_name", "title"]
flagged_cols = [c for c in df.columns
                 if any(t in c.lower() for t in sensitive_terms)
                 and "hash" not in c.lower() and "id" not in c.lower()]
print("Columns needing extra care (should be empty):", flagged_cols)
print("\nAll columns are pseudonymized/observable-only:")
print(list(df.columns))

Columns needing extra care (should be empty): []

All columns are pseudonymized/observable-only:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.